In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import numpy as np
import pandas as pd
import pickle
from datasets import load_dataset, Dataset, load_from_disk
import ast
from datasets import load_dataset, Dataset, DatasetDict
from dataclasses import dataclass, field
from typing import Optional
from tqdm import tqdm
from trl import SFTTrainer
import random
import wandb
tqdm.pandas()


import warnings
warnings.filterwarnings('ignore')

In [ ]:
################ Load Data + Process Data
print("START LOAD DATA")
path = 'data/physionet.org/files/clinical-bert-mimic-notes/1.0.0/setup_outputs/'
#icd9_descript = pd.read_csv('physionet.org/files/clinical-bert-mimic-notes/1.0.0/setup_outputs/ICD9_Descriptions.csv')
medcat_descript = pd.read_csv(path + 'MedCAT_Descriptions.csv')
reidentified_subject_ids = pd.read_csv(path + 'reidentified_subject_ids.csv')
#subject_id_to_icd9 = pd.read_csv('physionet.org/files/clinical-bert-mimic-notes/1.0.0/setup_outputs/SUBJECT_ID_to_ICD9.csv')
subject_id_to_medcat = pd.read_csv(path + 'SUBJECT_ID_to_MedCAT.csv')
subject_id_to_name = pd.read_csv(path + 'SUBJECT_ID_to_NAME.csv')
subject_id_to_notes_1a = pd.read_csv(path + 'SUBJECT_ID_to_NOTES_1a.csv')
subject_id_to_notes_1b = pd.read_csv(path + 'SUBJECT_ID_to_NOTES_1b.csv')
subject_id_to_notes_templates = pd.read_csv(path + 'SUBJECT_ID_to_NOTES_templates.csv')


with open(file='data/physionet.org/processed_data/train_data.pickle', mode='rb') as f:
    train_data = pickle.load(f)

with open(file='data/physionet.org/processed_data/test_data.pickle', mode='rb') as f:
    test_data = pickle.load(f)
print("DONE LOAD DATA")
    
medcat_name_subject = pd.merge(subject_id_to_medcat, medcat_descript, on='CODE', how='inner')
subject_medcat_names = medcat_name_subject.groupby(['SUBJECT_ID'])['DESCRIPTION'].apply(list).reset_index(name='code_name')


In [ ]:
def preprocessing_data(data, df):
    data_new = [list(x) for x in (data[:][0])]

    note = []
    subject_id = []

    for i in data_new:
        note.append(i[0])
        subject_id.append(i[1])

    df_train = pd.DataFrame({'note': note,'SUBJECT_ID': subject_id})

    df_train_merged = df_train.merge(df[['SUBJECT_ID', 'code_name']], on='SUBJECT_ID', how='left')

    return df_train_merged


def processing_df(df_data):
    #df_data['code_name']= df_data['code_name'].apply(lambda x: (ast.literal_eval(x)))
    df_data['condition_nums'] = df_data['code_name'].apply(lambda x: len(x))

    subject_id_to_name['FULL_NAME'] = subject_id_to_name['FIRST_NAME'] + ' ' + subject_id_to_name['LAST_NAME']
    fin_df = pd.merge(subject_id_to_name[['SUBJECT_ID', 'FULL_NAME']], df_data, on='SUBJECT_ID', how='inner')

    fin_df = fin_df[['SUBJECT_ID', 'FULL_NAME', 'condition_nums', 'note', 'code_name']]
    fin_df.columns = ['SUBJECT_ID', 'name', 'num', 'note', 'condition']

    fin_df['condition'] = fin_df['condition'].apply(lambda x: ", ".join(x))

    return fin_df

In [ ]:

print("START preprocessing_data(train_data, subject_medcat_names)")
train_df = preprocessing_data(train_data, subject_medcat_names)

print("START preprocessing_data(train_data, subject_medcat_names)")
test_df = preprocessing_data(test_data, subject_medcat_names)

print("START processing_df(train_df)")
df_train = processing_df(train_df)

print("START processing_df(test_df)")
df_test = processing_df(test_df)

In [ ]:
sample_c = df_train.sample(n=4000, replace=False, random_state=42)[['SUBJECT_ID', 'name', 'condition', 'num']]
sample_c.columns = ['SUBJECT_ID', 'name', 'condition', 'condition_nums']
sample_c

In [ ]:
sample_c.to_csv('sample_c.csv', index=False)